In [1]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

In [2]:
from src.pipeline.config import (
	TRAIN_TRANSACTION_PATH,
	TRAIN_IDENTITY_PATH,
	PROCESSED_DATA_DIR,
	RANDOM_SEED
)

In [3]:
df_trans = pd.read_csv(TRAIN_TRANSACTION_PATH, engine="pyarrow")
df_id = pd.read_csv(TRAIN_IDENTITY_PATH, engine="pyarrow")

print(f"Transaction shape: {df_trans.shape}")
print(f"Identity shape: {df_id.shape}")

Transaction shape: (590540, 394)
Identity shape: (144233, 41)


In [4]:
df = pd.merge(df_trans, df_id, on="TransactionID", how="left")
print(f"Final df shape: {df.shape}")

Final df shape: (590540, 434)


In [5]:
processed_path = PROCESSED_DATA_DIR / "train_merged.parquet"
df.to_parquet(processed_path, engine="pyarrow")
print(f"Saved in {processed_path}")

Saved in /home/arcsin/IEEE_CIS/data/processed/train_merged.parquet


In [6]:
df.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [7]:
print(f"RAM usage: {df.memory_usage().sum() / 1024**2:2f} MB")
print(df.dtypes.value_counts())

RAM usage: 1984.064550 MB
float64    399
str         31
int64        4
Name: count, dtype: int64


In [8]:
df[["TransactionDT", "TransactionAmt"]].describe()

,TransactionDT,TransactionAmt
count,5.905400e+05,590540.000000
mean,7.372311e+06,135.027176
std,4.617224e+06,239.162522
min,8.640000e+04,0.251000
25%,3.027058e+06,43.321000
50%,7.306528e+06,68.769000
75%,1.124662e+07,125.000000
max,1.581113e+07,31937.391000


In [9]:
print(df['isFraud'].value_counts(normalize=True) * 100)

isFraud
0    96.500999
1     3.499001
Name: proportion, dtype: float64


In [10]:
X = df.drop(["isFraud", "DeviceInfo"], axis=1)
y = df["isFraud"]

In [11]:
float_cols = X.select_dtypes(include=['float64']).columns
X[float_cols] = X[float_cols].astype('float32')

In [12]:
cat_features = X.select_dtypes(exclude=['number']).columns.tolist()
X[cat_features] = X[cat_features].fillna('missing').astype(str)
print(X[cat_features].nunique().sort_values(ascending=False))

id_33            261
id_31            131
id_30             76
R_emaildomain     61
P_emaildomain     60
card4              5
id_34              5
ProductCD          5
card6              5
id_23              4
id_15              4
M4                 4
M1                 3
M2                 3
M9                 3
M8                 3
M7                 3
M6                 3
M5                 3
M3                 3
id_16              3
id_12              3
id_29              3
id_28              3
id_27              3
id_35              3
id_36              3
id_37              3
id_38              3
DeviceType         3
dtype: int64


In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
	X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)

In [14]:
len(cat_features)

30

In [15]:
from catboost import CatBoostClassifier

clf = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    random_seed=RANDOM_SEED,
    custom_loss="AUC"
)

In [16]:
import gc
gc.collect()

clf.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    verbose=10
)

0:	learn: 0.5373039	test: 0.5373215	best: 0.5373215 (0)	total: 736ms	remaining: 3m 40s
10:	learn: 0.1395944	test: 0.1397772	best: 0.1397772 (10)	total: 5.4s	remaining: 2m 21s
20:	learn: 0.1092563	test: 0.1095985	best: 0.1095985 (20)	total: 9.88s	remaining: 2m 11s
30:	learn: 0.1025476	test: 0.1031620	best: 0.1031620 (30)	total: 14s	remaining: 2m 1s
40:	learn: 0.0987914	test: 0.0996452	best: 0.0996452 (40)	total: 17.8s	remaining: 1m 52s
50:	learn: 0.0965943	test: 0.0974198	best: 0.0974198 (50)	total: 21.7s	remaining: 1m 45s
60:	learn: 0.0944915	test: 0.0954266	best: 0.0954266 (60)	total: 25.5s	remaining: 1m 39s
70:	learn: 0.0928350	test: 0.0938629	best: 0.0938629 (70)	total: 29.4s	remaining: 1m 34s
80:	learn: 0.0915236	test: 0.0926988	best: 0.0926988 (80)	total: 33.8s	remaining: 1m 31s
90:	learn: 0.0902198	test: 0.0914142	best: 0.0914142 (90)	total: 37.9s	remaining: 1m 26s
100:	learn: 0.0891404	test: 0.0903042	best: 0.0903042 (100)	total: 41.8s	remaining: 1m 22s
110:	learn: 0.0881655	tes

CatBoostClassifier(custom_loss='AUC', iterations=300, learning_rate=0.1, random_seed=42)

In [17]:
from sklearn.metrics import roc_auc_score

val_preds = clf.predict_proba(X_test)[:, 1]

auc_score = roc_auc_score(y_test, val_preds)
print(f"Baseline ROC-AUC: {auc_score:.5f}")

Baseline ROC-AUC: 0.91214


In [18]:
feature_imp = pd.Series(clf.get_feature_importance(), index=X_train.columns)
print(feature_imp.sort_values(ascending=False).head(10))

C14               4.564695
M4                4.033837
C13               3.655771
TransactionAmt    3.516973
C1                3.174625
V308              2.851778
card6             2.699567
V317              2.152285
card2             2.045798
C11               2.015852
dtype: float64
